# 05 — Inference of 4 Models over All Dota 2 Chat (2016-2026)

Loop over {BERT, RoBERTa, DistilBERT} × {sentiment, toxicity} + Detoxify (toxicity, zero-shot) across every `data/processed/<folder>.parquet`.

Output: `data/inference/<model>_<task>/<folder>.parquet` with columns `match_id, time, player_slot, prob_*, pred_*`.

**Slang parity:** inference applies the *same* `translate_slang` preprocessing used in training (notebook 04). Controlled by `preprocessing.slang_translation` in `configs/experiment.yaml`. If train and inference disagree on this flag, predictions degrade.

**Requires a CUDA GPU.** ~2-8 h for the full decade. Skip-if-exists per folder so it can resume.

**Prerequisite:** notebook 04 finished (checkpoints in `models/`).

> ⚠️ **Re-running after the in-domain rebuild:** inference is *skip-if-exists*. The old `data/inference/*.parquet` were produced by the previous external-trained models — they are **stale**. Delete them (and retrain `models/` via NB04) before re-running, or you will keep the worse-than-chance predictions:
> ```python
> import shutil; shutil.rmtree('data/inference', ignore_errors=True)
> ```

In [1]:
%pip install detoxify
!pip install detoxify

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Sel 1: Setup
import sys
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'src').is_dir() and (p / 'configs').is_dir()), _here)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

from src.runtime import load_config, print_banner, RunLog
config = load_config('configs/experiment.yaml')
print_banner('05_inference', config)
run_log = RunLog(notebook='05_inference', config_path='configs/experiment.yaml')

import torch
print(f'CUDA: {torch.cuda.is_available()}')

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook: 05_inference
Experiment: thesis-sentiment-toxicity-dota2-decade
Seed: 42  |  Git: da158c0
Started at: 2026-06-10T06:10:49+00:00Z
Versi paket:
  - python: 3.10.0
  - transformers: 5.5.0
  - torch: 2.8.0+cu128
  - datasets: 4.8.5
  - scikit-learn: 1.7.1
  - pandas: 2.3.3
  - numpy: 1.26.4
CUDA: True


In [3]:
# Sel 2: Configuration
PROCESSED_ROOT = Path(config['data']['processed_root'])
INFERENCE_ROOT = Path(config['data']['inference_root'])
MODELS_ROOT = Path('models')
BATCH_SIZE = int(config['inference']['batch_size'])
FP16 = config['inference']['precision'] == 'fp16'

# Slang parity with training (notebook 04). MUST match the flag used there.
APPLY_SLANG = bool(config.get('preprocessing', {}).get('slang_translation', True))
print(f'apply_slang (train/infer parity) = {APPLY_SLANG}')

FT_MODELS = ['bert', 'roberta', 'distilbert']

processed_files = sorted([p for p in PROCESSED_ROOT.glob('*.parquet') if not p.stem.endswith('_non_english')])
print(f'Processed folders: {len(processed_files)}')
for p in processed_files:
    print(f'  - {p.name}')

apply_slang (train/infer parity) = True
Processed folders: 14
  - 2016.parquet
  - 2017.parquet
  - 2018.parquet
  - 2019.parquet
  - 2020.parquet
  - 2021.parquet
  - 2022.parquet
  - 2023.parquet
  - 2024.parquet
  - 2025.parquet
  - 202601.parquet
  - 202602.parquet
  - 202603.parquet
  - 202604.parquet


In [4]:
# Sel 3: Inference — 3 fine-tuned models × 2 tasks
from src.inference.batch_inference import infer_folder

for model_key in FT_MODELS:
    for task in ['sentiment', 'toxicity']:
        model_dir = MODELS_ROOT / f'{model_key}-{task}'
        if not model_dir.exists():
            msg = f'Skip {model_key}/{task} — checkpoint missing at {model_dir}'
            print(f'[WARN] {msg}')
            run_log.add_warning(msg)
            continue
        out_root = INFERENCE_ROOT / f'{model_key}_{task}'
        out_root.mkdir(parents=True, exist_ok=True)
        print(f'\n=== {model_key} / {task} ({model_dir}) ===')
        for p in processed_files:
            out_path = out_root / p.name
            stats = infer_folder(
                processed_path=p,
                model_dir=model_dir,
                task=task,
                out_path=out_path,
                batch_size=BATCH_SIZE,
                fp16=FP16,
                apply_slang=APPLY_SLANG,
            )
            tag = '[SKIP]' if stats.skipped else '[OK]'
            print(f'  {tag} {p.name}: n={stats.n_output:,} → {stats.out_path}')
            run_log.add_output(stats.out_path)


=== bert / sentiment (models\bert-sentiment) ===
  [SKIP] 2016.parquet: n=120,454 → data\inference\bert_sentiment\2016.parquet
  [SKIP] 2017.parquet: n=101,812 → data\inference\bert_sentiment\2017.parquet
  [SKIP] 2018.parquet: n=114,866 → data\inference\bert_sentiment\2018.parquet
  [SKIP] 2019.parquet: n=187,072 → data\inference\bert_sentiment\2019.parquet
  [SKIP] 2020.parquet: n=163,577 → data\inference\bert_sentiment\2020.parquet
  [SKIP] 2021.parquet: n=152,358 → data\inference\bert_sentiment\2021.parquet
  [SKIP] 2022.parquet: n=157,225 → data\inference\bert_sentiment\2022.parquet
  [SKIP] 2023.parquet: n=170,543 → data\inference\bert_sentiment\2023.parquet
  [SKIP] 2024.parquet: n=182,574 → data\inference\bert_sentiment\2024.parquet
  [SKIP] 2025.parquet: n=197,781 → data\inference\bert_sentiment\2025.parquet
  [SKIP] 202601.parquet: n=18,423 → data\inference\bert_sentiment\202601.parquet
  [SKIP] 202602.parquet: n=13,442 → data\inference\bert_sentiment\202602.parquet
  [SKIP]

In [5]:
# Sel 4: Inference — Detoxify (zero-shot, no fine-tune). Slang applied for fair comparison.
from src.inference.batch_inference import infer_detoxify

out_root = INFERENCE_ROOT / 'detoxify_toxicity'
out_root.mkdir(parents=True, exist_ok=True)
print(f'\n=== detoxify / toxicity (pretrained, zero-shot) ===')
for p in processed_files:
    out_path = out_root / p.name
    stats = infer_detoxify(
        processed_path=p,
        out_path=out_path,
        batch_size=BATCH_SIZE,
        apply_slang=APPLY_SLANG,
    )
    tag = '[SKIP]' if stats.skipped else '[OK]'
    print(f'  {tag} {p.name}: n={stats.n_output:,}')
    run_log.add_output(stats.out_path)


=== detoxify / toxicity (pretrained, zero-shot) ===
  [SKIP] 2016.parquet: n=120,454
  [SKIP] 2017.parquet: n=101,812
  [SKIP] 2018.parquet: n=114,866
  [SKIP] 2019.parquet: n=187,072
  [SKIP] 2020.parquet: n=163,577
  [SKIP] 2021.parquet: n=152,358
  [SKIP] 2022.parquet: n=157,225
  [SKIP] 2023.parquet: n=170,543
  [SKIP] 2024.parquet: n=182,574
  [SKIP] 2025.parquet: n=197,781
  [SKIP] 202601.parquet: n=18,423
  [SKIP] 202602.parquet: n=13,442
  [SKIP] 202603.parquet: n=14,087
  [SKIP] 202604.parquet: n=9,353


In [6]:
# Sel 5: Verifikasi join lossless inference × processed
import pandas as pd

# Cek satu kombinasi sebagai sanity check.
for model_key in FT_MODELS:
    for task in ['sentiment', 'toxicity']:
        out_root = INFERENCE_ROOT / f'{model_key}_{task}'
        if not out_root.exists():
            continue
        inf_files = sorted(out_root.glob('*.parquet'))
        proc_total = sum(len(pd.read_parquet(p)) for p in processed_files)
        inf_total = sum(len(pd.read_parquet(p)) for p in inf_files)
        if proc_total != inf_total:
            msg = f'Mismatch {model_key}/{task}: processed={proc_total:,} vs inference={inf_total:,}'
            print(f'[WARN] {msg}')
            run_log.add_warning(msg)
        else:
            print(f'  OK {model_key}/{task}: {inf_total:,} baris cocok dengan processed')

  OK bert/sentiment: 1,603,567 baris cocok dengan processed
  OK bert/toxicity: 1,603,567 baris cocok dengan processed
  OK roberta/sentiment: 1,603,567 baris cocok dengan processed
  OK roberta/toxicity: 1,603,567 baris cocok dengan processed
  OK distilbert/sentiment: 1,603,567 baris cocok dengan processed
  OK distilbert/toxicity: 1,603,567 baris cocok dengan processed


In [7]:
run_log.save('reports/run_log.csv')

[run_log] 05_inference → 11.86s, 98 outputs, 0 warnings → reports\run_log.csv
